In [ ]:
CODE_DIR = "/kaggle/input/kavishahaswani/occluded-object-detection-code"
MOCA = "/kaggle/input/ivanomelchenkoim11/moca-dataset/MoCA"
CAD = "/kaggle/input/tanushkalokegaonkar/camouflaged-animal-dataset/CamouflagedAnimalDataset"
YUV = "/kaggle/input/tanushkalokegaonkar/yuv20k/YUV20K"
print("CODE:",CODE_DIR,"| MOCA:",MOCA,"| CAD:",CAD,"| YUV:",YUV)


In [ ]:
!python {CODE_DIR}/make_split.py --moca_root /kaggle/working/moca_converted --cad_root {CAD} --yuv_root {YUV} --out /kaggle/working/splits
!cat /kaggle/working/splits/split_summary.json 2>&1 | head -c 400


In [ ]:
!python {CODE_DIR}/convert_moca_csv_to_masks.py --moca_root {MOCA} --output_root /kaggle/working/moca_converted


In [ ]:
!mkdir -p /kaggle/working/cache_amp /kaggle/working/runs_amp
!python {CODE_DIR}/preprocess_amplify.py --data_roots moca=/kaggle/working/moca_converted cad={CAD} yuv20k={YUV} --cache_root /kaggle/working/cache_amp --fps 24 --window_size 40 --overlap 20 --motion_size 256
!python {CODE_DIR}/train.py --data_roots moca=/kaggle/working/moca_converted cad={CAD} yuv20k={YUV} --adapters moca=moca_mask cad=cad yuv20k=imgs_gt --cache_root /kaggle/working/cache_amp --split_dir /kaggle/working/splits --split train --checkpoint_dir /kaggle/working/runs_amp --epochs 10 --batch_size 8 --image_size 256 --base_ch 32 --seed 42


In [ ]:
!mkdir -p /kaggle/working/cache_raw /kaggle/working/runs_raw
!python {CODE_DIR}/preprocess_amplify.py --data_roots moca=/kaggle/working/moca_converted cad={CAD} yuv20k={YUV} --cache_root /kaggle/working/cache_raw --fps 24 --window_size 40 --overlap 20 --motion_size 256 --no_amplify
!python {CODE_DIR}/train.py --data_roots moca=/kaggle/working/moca_converted cad={CAD} yuv20k={YUV} --adapters moca=moca_mask cad=cad yuv20k=imgs_gt --cache_root /kaggle/working/cache_raw --split_dir /kaggle/working/splits --split train --checkpoint_dir /kaggle/working/runs_raw --epochs 10 --batch_size 8 --image_size 256 --base_ch 32 --seed 42
